ASSIGNMENT NLP – 4 Fine-Tuning BERT on a Kaggle Dataset


In [ ]:
!pip install transformers datasets

In [ ]:
import pandas as pd
import numpy as np
import torch

In [ ]:
df = pd.read_csv("IMDB Dataset.csv", encoding='latin1', on_bad_lines='skip', engine='python')

In [ ]:
df.head()
df.info()
df['sentiment'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22150 entries, 0 to 22149
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     22150 non-null  object
 1   sentiment  22150 non-null  object
dtypes: object(2)
memory usage: 346.2+ KB


,count
sentiment,
negative,11137
positive,11013


In [ ]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

In [ ]:
df.dropna(inplace=True)

In [ ]:
df['review'] = df['review'].str.lower()
df['review'] = df['review'].str.strip()

In [ ]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. <br /><br />the...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,"petter mattei's ""love in the time of money"" is...",1


In [ ]:
import re

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)  # remove HTML
    return text

df['review'] = df['review'].apply(clean_text)

In [ ]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. the filming tec...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,"petter mattei's ""love in the time of money"" is...",1


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=df['sentiment']
)

In [ ]:
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

In [ ]:
print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))

Train size: 17720
Validation size: 2215
Test size: 2215


In [ ]:
print(train_labels.value_counts(normalize=True))
print(val_labels.value_counts(normalize=True))
print(test_labels.value_counts(normalize=True))

sentiment
0    0.502822
1    0.497178
Name: proportion, dtype: float64
sentiment
0    0.502483
1    0.497517
Name: proportion, dtype: float64
sentiment
0    0.502935
1    0.497065
Name: proportion, dtype: float64


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
sample_text = train_texts.iloc[0]

tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Text:", sample_text[:100])
print("Tokens:", tokens[:20])
print("Token IDs:", token_ids[:20])

Text: a chinese scholar who criticizes harshly the arrogant nationalist, warmongering policies of the ruli
Tokens: ['a', 'chinese', 'scholar', 'who', 'critic', '##izes', 'harshly', 'the', 'arrogant', 'nationalist', ',', 'warm', '##ong', '##ering', 'policies', 'of', 'the', 'ruling', 'cl', '##ique']
Token IDs: [1037, 2822, 6288, 2040, 6232, 10057, 21052, 1996, 15818, 8986, 1010, 4010, 5063, 7999, 6043, 1997, 1996, 6996, 18856, 7413]


In [ ]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
print(train_encodings.keys())
print(len(train_encodings['input_ids']))
print(len(train_encodings['attention_mask']))

Buffered data was truncated after reaching the output size limit.

In [ ]:
print(train_encodings['input_ids'][0][:20])
print(train_encodings['attention_mask'][0][:20])

[101, 1037, 2822, 6288, 2040, 6232, 10057, 21052, 1996, 15818, 8986, 1010, 4010, 5063, 7999, 6043, 1997, 1996, 6996, 18856]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
